# Week 2: Baseline Models Evaluation

This notebook trains and evaluates the baseline models for the SD-MoSE project.

## Models
1. **Linear Regression** (Global)
2. **Latitude-Band Linear Regression** (Regional)
3. **Random Forest** (Non-linear Upper Bound)
4. **XGBoost** (Gradient Boosting Upper Bound)
5. **Hard K-Means + Symbolic Regression** (Current Best Interpretable)

## Metrics
- RMSE (Root Mean Squared Error)
- R² Score

In [1]:
# Setup
import sys
import os
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

sys.path.append(os.path.abspath('..'))

from scripts.data import TRAIN_OUTPUT_PATH, TEST_OUTPUT_PATH
from src.climate_discovery.models.baselines import (
    LinearBaseline, 
    LatitudeBandLinearRegression, 
    RFBaseline, 
    XGBBaseline
)
from src.climate_discovery.models.symbolic import KMeansSymbolicRegressor

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


## 1. Load & Prepare Data

In [2]:
# Load NetCDF
ds_train = xr.open_dataset(TRAIN_OUTPUT_PATH)
ds_test = xr.open_dataset(TEST_OUTPUT_PATH)

# Function to flatten XArray to (N, Features)
def to_tabular(ds, target_var='fco2'):
    # Select features
    feature_vars = ['sst', 'sss', 'sin_month', 'cos_month', 'year_feature'] # Using Physics + Time first (No Bio yet for fair comparison? Or use Bio?)
    # Let's use ALL features available in preprocess including log_chl
    feature_vars = ['sst', 'sss', 'log_chl', 'sin_month', 'cos_month', 'year_feature']
    
    # Stack (Time, Lat, Lon) -> (Sample)
    df = ds.to_dataframe().reset_index().dropna()
    
    X = df[feature_vars].values
    y = df[target_var].values
    lat = df['lat'].values # For Lat-Band model
    
    return X, y, lat, df

print("Converting to Tabular...")
X_train, y_train, lat_train, df_train = to_tabular(ds_train)
X_test, y_test, lat_test, df_test = to_tabular(ds_test)

print(f"Train Samples: {X_train.shape[0]}")
print(f"Test Samples: {X_test.shape[0]}")

Converting to Tabular...
Train Samples: 98373
Test Samples: 31193


## 2. Train Models

In [3]:
results = {}
models = {}

# 1. Linear Regression
print("Training Linear Regression...")
model_lr = LinearBaseline()
model_lr.fit(X_train, y_train)
models['Linear'] = model_lr

# 2. Lat-Band Linear
print("Training Lat-Band Linear...")
model_lat = LatitudeBandLinearRegression()
model_lat.fit(X_train, y_train, lat_train)
models['Lat-Band LR'] = model_lat

# 3. Random Forest (Subsampled for speed if needed)
print("Training Random Forest (this may take a while)...")
# Using modest params for baseline speed
model_rf = RFBaseline(n_estimators=50, max_depth=15, n_jobs=-1)
model_rf.fit(X_train, y_train)
models['RandomForest'] = model_rf

# 4. XGBoost
print("Training XGBoost...")
model_xgb = XGBBaseline(n_estimators=100, max_depth=6)
model_xgb.fit(X_train, y_train)
models['XGBoost'] = model_xgb

# 5. KMeans + Symbolic (The previous best)
print("Training KMeans + Symbolic...")
model_sym = KMeansSymbolicRegressor(n_clusters=5, max_depth=4)
model_sym.fit(X_train, y_train)
models['KMeans+Symbolic'] = model_sym

Training Linear Regression...
Training Lat-Band Linear...
Training Random Forest (this may take a while)...
Training XGBoost...
Training KMeans + Symbolic...
Fitting Symbolic Expert for Cluster 0 (42581 samples)...


c:\Users\shlok\OneDrive\Desktop\Projects\climate-equation-discovery\venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
c:\Users\shlok\OneDrive\Desktop\Projects\climate-equation-discovery\venv\Lib\site-packages\pysr\sr.py:2265: UserWarning: Note: you are running with more than 10,000 datapoints. You should consider turning on batching (https://ai.damtp.cam.ac.uk/pysr/options/#batching). You should also reconsider if you need that many datapoints. Unless you have a large amount of noise (in which case you should smooth your dataset first), generally < 10,000 datapoints is enough to find a functional form with symbolic regression. More datapoints will lower the search speed.
  warnings.warn(
c:\Users\shlok\OneDrive\Desktop\Projects\climate-equation-discovery\venv\Lib\site-packages\pysr\sr.py:1873: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `p

Fitting Symbolic Expert for Cluster 1 (12900 samples)...


c:\Users\shlok\OneDrive\Desktop\Projects\climate-equation-discovery\venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
c:\Users\shlok\OneDrive\Desktop\Projects\climate-equation-discovery\venv\Lib\site-packages\pysr\sr.py:2265: UserWarning: Note: you are running with more than 10,000 datapoints. You should consider turning on batching (https://ai.damtp.cam.ac.uk/pysr/options/#batching). You should also reconsider if you need that many datapoints. Unless you have a large amount of noise (in which case you should smooth your dataset first), generally < 10,000 datapoints is enough to find a functional form with symbolic regression. More datapoints will lower the search speed.
  warnings.warn(
c:\Users\shlok\OneDrive\Desktop\Projects\climate-equation-discovery\venv\Lib\site-packages\pysr\sr.py:1873: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `p

Fitting Symbolic Expert for Cluster 2 (23225 samples)...


c:\Users\shlok\OneDrive\Desktop\Projects\climate-equation-discovery\venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
c:\Users\shlok\OneDrive\Desktop\Projects\climate-equation-discovery\venv\Lib\site-packages\pysr\sr.py:2265: UserWarning: Note: you are running with more than 10,000 datapoints. You should consider turning on batching (https://ai.damtp.cam.ac.uk/pysr/options/#batching). You should also reconsider if you need that many datapoints. Unless you have a large amount of noise (in which case you should smooth your dataset first), generally < 10,000 datapoints is enough to find a functional form with symbolic regression. More datapoints will lower the search speed.
  warnings.warn(
c:\Users\shlok\OneDrive\Desktop\Projects\climate-equation-discovery\venv\Lib\site-packages\pysr\sr.py:1873: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `p

Fitting Symbolic Expert for Cluster 3 (955 samples)...


c:\Users\shlok\OneDrive\Desktop\Projects\climate-equation-discovery\venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
c:\Users\shlok\OneDrive\Desktop\Projects\climate-equation-discovery\venv\Lib\site-packages\pysr\sr.py:1873: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(


Fitting Symbolic Expert for Cluster 4 (18712 samples)...


c:\Users\shlok\OneDrive\Desktop\Projects\climate-equation-discovery\venv\Lib\site-packages\pysr\sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
c:\Users\shlok\OneDrive\Desktop\Projects\climate-equation-discovery\venv\Lib\site-packages\pysr\sr.py:2265: UserWarning: Note: you are running with more than 10,000 datapoints. You should consider turning on batching (https://ai.damtp.cam.ac.uk/pysr/options/#batching). You should also reconsider if you need that many datapoints. Unless you have a large amount of noise (in which case you should smooth your dataset first), generally < 10,000 datapoints is enough to find a functional form with symbolic regression. More datapoints will lower the search speed.
  warnings.warn(
c:\Users\shlok\OneDrive\Desktop\Projects\climate-equation-discovery\venv\Lib\site-packages\pysr\sr.py:1873: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `p

## 3. Evaluation

In [4]:
metrics = []

for name, model in models.items():
    print(f"Evaluating {name}...")
    
    # Predict
    if name == 'Lat-Band LR':
        y_pred = model.predict(X_test, lat_test)
    else:
        y_pred = model.predict(X_test)
        
    # Calculate Metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    metrics.append({
        "Model": name,
        "RMSE": rmse,
        "R2": r2
    })

metrics_df = pd.DataFrame(metrics).sort_values("R2", ascending=False)
display(metrics_df)

Evaluating Linear...
Evaluating Lat-Band LR...
Evaluating RandomForest...
Evaluating XGBoost...
Evaluating KMeans+Symbolic...


,Model,RMSE,R2
2,RandomForest,34.755392,0.526240
3,XGBoost,35.692837,0.500338
1,Lat-Band LR,39.781821,0.379298
4,KMeans+Symbolic,42.482760,0.292153
0,Linear,46.222541,0.162043


## 4. Visualize Symbolic Equations

In [5]:
print("Discovered Equations (KMeans+Symbolic):")
eqs = models['KMeans+Symbolic'].get_equations()
for cluster, eq in eqs.items():
    print(f"{cluster}: {eq}")

Discovered Equations (KMeans+Symbolic):
Cluster 0: (x0 * 21.568634) + 372.46863
Cluster 1: (x4 * 27.929478) + 345.64023
Cluster 2: (x0 * 33.809345) + 364.07056
Cluster 3: (x4 + 2.4159858) * 151.87523
Cluster 4: ((sin(x4 + (x0 * x0)) - x2) / 0.08453932) + 372.40195
